In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# All auxiliary code is in ../src
import sys
sys.path.append("../src/")

In [3]:
RANDOM_STATE=42

In [4]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.ensemble import RandomForestClassifier
import shap
from sklearn.metrics import classification_report

# eXplainable AI

Since in the model selection phase of the previous task, we have identified a RandomForestClassifier (with a specific data preprocessing step) as the best classifier for our goals, let's apply the XAI techniques to it.

## Dataset generation

Let's reconstruct the classifier.

In [5]:
from classification_utility import train_test_for_non_distance, split_in_train_and_validation

In [6]:
train_set, test_set, train_label, test_label=train_test_for_non_distance()

Mapping of geo area: {'Africa': 0, 'America': 1, 'Asia': 2, 'Eastern Europe': 3, 'Northern Europe': 4, 'Oceania': 5, 'Southern Europe': 6, 'Western Europe': 7}


In [7]:
rand_oversampler=RandomOverSampler(random_state=RANDOM_STATE, sampling_strategy=0.67)
train_set_os, train_label_os = rand_oversampler.fit_resample(train_set, train_label)

In [8]:
print(f'TRAINING: {train_set_os.shape[0]} - {train_label_os.shape[0]}')
print(f'TEST: {test_set.shape[0]} - {test_label.shape[0]}')

TRAINING: 672056 - 672056
TEST: 32586 - 32586


In [9]:
TR_set_os, VD_set_os, TR_label_os, VD_label_os=split_in_train_and_validation(train_set_os, train_label_os)
print(f'TRAINING: {TR_set_os.shape[0]} - {TR_label_os.shape[0]}')
print(f'VALIDATION: {VD_set_os.shape[0]} - {VD_label_os.shape[0]}')
print(f'TEST: {test_set.shape[0]} - {test_label.shape[0]}')

TRAINING: 470439 - 470439
VALIDATION: 201617 - 201617
TEST: 32586 - 32586


In [10]:
#Custom function to print the classification report con training/valid
def print_report(train_label, train_predictions, validation_label, valid_predictions, v_or_t='TEST'):
    #v_or_t to print TEST or VALIDATION RESULTS
    classes=['not top20', 'top20']
    if train_label is not None:
        print('TRAINING SET RESULTS')
        print(classification_report(train_label, train_predictions, target_names=classes))
        print('--------------------')
    print(v_or_t+' SET RESULTS')
    print(classification_report(validation_label, valid_predictions, target_names=classes))

In [11]:
rf = RandomForestClassifier(n_estimators=30, 
                             criterion='entropy',
                             max_features='sqrt',
                             max_depth=30, 
                             min_samples_split=25,
                             min_samples_leaf=10,
                             bootstrap=True)
rf = rf.fit(train_set_os, train_label_os)
train_pred_rf = rf.predict(train_set_os)
test_pred_rf = rf.predict(test_set)
print_report(train_label=train_label_os, train_predictions=train_pred_rf, validation_label=test_label, valid_predictions=test_pred_rf)

TRAINING SET RESULTS
              precision    recall  f1-score   support

   not top20       0.92      0.93      0.93    402429
       top20       0.90      0.87      0.89    269627

    accuracy                           0.91    672056
   macro avg       0.91      0.90      0.91    672056
weighted avg       0.91      0.91      0.91    672056

--------------------
TEST SET RESULTS
              precision    recall  f1-score   support

   not top20       0.91      0.91      0.91     28066
       top20       0.43      0.42      0.42      4520

    accuracy                           0.84     32586
   macro avg       0.67      0.66      0.67     32586
weighted avg       0.84      0.84      0.84     32586



In [12]:
perturbation_data = train_set_os
perturbation_labels = train_label_os
perturbation_predictions = train_pred_rf

explanation_data = test_set
explanation_labels = test_label
explanation_predictions = test_pred_rf

In [13]:
interventional_explanation_algorithm = shap.TreeExplainer(
    model=rf,
    data=train_set_os,                       # perturb on a causal model induced on perturbation data
    feature_perturbation="interventional"  # use a causal model
)

distributional_explanation_algorithm = shap.TreeExplainer(
    model=rf,
    feature_perturbation="tree_path_dependent"  # condition on the distribution learned on the train data
)

: 

In [ ]:
try:
    print("Testing interventional algorithm...")
    interventional_explanations = interventional_explanation_algorithm(explanation_data)
    print(f"Interventional shape: {interventional_explanations.shape}")
except Exception as e:
    print(f"Error in interventional algorithm: {e}")

try:
    print("Testing distributional algorithm...")
    distributional_explanations = distributional_explanation_algorithm(explanation_data)
    print(f"Distributional shape: {distributional_explanations.shape}")
except Exception as e:
    print(f"Error in distributional algorithm: {e}")

In [14]:
interventional_explanations = interventional_explanation_algorithm(explanation_data)
distributional_explanations = distributional_explanation_algorithm(explanation_data)

print(interventional_explanations.shape)
print(distributional_explanations.shape)

#shap_interventional_explanations = interventional_explanations.values
#shap_distributional_explanations = distributional_explanations.values

: 